Cleaning up all datasets

Import pandas and initalizing what the final dataframe columns will be.

In [5]:
import pandas as pd

final_columns = ["text", "label", "source_dataset", "dataset_name"]

def clean_text_column(series):
    return (
        series.fillna("")
        .astype(str)
        .str.strip()
    )

def standardize_dataset(df, text_cols, label_col, label_map, source_name, dataset_name):
    df = df.copy()

    # combine one or more text columns into a single text column
    available_text_cols = [col for col in text_cols if col in df.columns]
    if not available_text_cols:
        raise KeyError(f"None of the text columns were found: {text_cols}")

    text_parts = [clean_text_column(df[col]) for col in available_text_cols]
    df["text"] = text_parts[0]
    for part in text_parts[1:]:
        df["text"] = df["text"] + " " + part

    df["text"] = df["text"].str.strip()

    # standardize label
    label_series = df[label_col]
    if pd.api.types.is_numeric_dtype(label_series):
        normalized_labels = pd.to_numeric(label_series, errors="coerce")
        normalized_map = label_map
    else:
        normalized_labels = label_series.astype(str).str.strip().str.lower()
        normalized_map = {
            str(key).strip().lower(): value
            for key, value in label_map.items()
        }

    df["label"] = normalized_labels.map(normalized_map)
    unmatched_labels = sorted(pd.Series(normalized_labels[df["label"].isna()]).dropna().unique().tolist())
    if unmatched_labels:
        print(f"Unmatched labels for {source_name}: {unmatched_labels[:10]}")

    # add source name
    df["source_dataset"] = source_name
    df["dataset_name"] = dataset_name

    # keep only needed columns
    df = df[final_columns]

    # remove bad rows
    df = df.dropna(subset=["text", "label"])
    df = df[df["text"] != ""]

    # optional: force label to int
    df["label"] = df["label"].astype(int)

    # remove duplicates based on text
    df = df.drop_duplicates(subset=["text"])

    return df

def sample_balanced_by_dataset(df, target_per_label=7500, random_state=42):
    sampled_parts = []

    for label_value in sorted(df["label"].unique()):
        label_df = df[df["label"] == label_value].copy()
        if len(label_df) < target_per_label:
            raise ValueError(
                f"Not enough rows for label {label_value}: "
                f"{len(label_df)} available, {target_per_label} requested"
            )

        counts = label_df["dataset_name"].value_counts().sort_index()
        raw_targets = counts / counts.sum() * target_per_label
        base_targets = raw_targets.astype(int)
        remainders = raw_targets - base_targets

        if (counts >= 1).all() and len(base_targets) <= target_per_label:
            base_targets = base_targets.mask(base_targets == 0, 1)

        overflow = int(base_targets.sum() - target_per_label)
        if overflow > 0:
            for dataset_name in base_targets.sort_values(ascending=False).index:
                reducible = min(overflow, max(0, base_targets[dataset_name] - 1))
                if reducible > 0:
                    base_targets[dataset_name] -= reducible
                    overflow -= reducible
                if overflow == 0:
                    break

        shortfall = int(target_per_label - base_targets.sum())
        if shortfall > 0:
            for dataset_name in remainders.sort_values(ascending=False).index:
                available_extra = counts[dataset_name] - base_targets[dataset_name]
                if available_extra > 0:
                    base_targets[dataset_name] += 1
                    shortfall -= 1
                if shortfall == 0:
                    break

        for dataset_name, n_rows in base_targets.items():
            if n_rows > 0:
                subset = label_df[label_df["dataset_name"] == dataset_name]
                sampled_parts.append(subset.sample(n=int(n_rows), random_state=random_state))

    sampled_df = pd.concat(sampled_parts, ignore_index=True)
    return sampled_df.sample(frac=1, random_state=random_state).reset_index(drop=True)


Load, clean, and combine the requested datasets using the reusable helper functions.

In [6]:
fake_news_test = standardize_dataset(
    pd.read_csv("Data/fake_news_test.csv"),
    text_cols=["text"],
    label_col="label",
    label_map={"Fake": 1, "Real": 0},
    source_name="news",
    dataset_name="fake_news_test"
)

fake_news_train = standardize_dataset(
    pd.read_csv("Data/fake_news_train.csv"),
    text_cols=["text"],
    label_col="label",
    label_map={"Fake": 1, "Real": 0},
    source_name="news",
    dataset_name="fake_news_train"
)

train = standardize_dataset(
    pd.read_csv("Data/train.csv"),
    text_cols=["text"],
    label_col="target",
    label_map={0: 1, 1: 0},
    source_name="twitter",
    dataset_name="train"
)

train1 = standardize_dataset(
    pd.read_csv("Data/train1.csv"),
    text_cols=["text"],
    label_col="label",
    label_map={"fake": 1, "real": 0},
    source_name="news",
    dataset_name="train1"
)

true_news = standardize_dataset(
    pd.read_csv("Data/True.csv").assign(label=0),
    text_cols=["text"],
    label_col="label",
    label_map={0: 0},
    source_name="news",
    dataset_name="True"
)

fake_news_all = standardize_dataset(
    pd.read_csv("Data/Fake.csv").assign(label=1),
    text_cols=["text"],
    label_col="label",
    label_map={1: 1},
    source_name="news",
    dataset_name="Fake"
)

train_tsv_raw = pd.read_csv("Data/train.tsv", sep="\t", header=None)
train_tsv_raw = train_tsv_raw[train_tsv_raw[1].isin(["true", "false"])]
train_tsv = standardize_dataset(
    train_tsv_raw.rename(columns={2: "text", 1: "label"}),
    text_cols=["text"],
    label_col="label",
    label_map={"true": 0, "false": 1},
    source_name="twitter",
    dataset_name="train_tsv"
)

test_tsv_raw = pd.read_csv("Data/test.tsv", sep="\t", header=None)
test_tsv_raw = test_tsv_raw[test_tsv_raw[1].isin(["true", "false"])]
test_tsv = standardize_dataset(
    test_tsv_raw.rename(columns={2: "text", 1: "label"}),
    text_cols=["text"],
    label_col="label",
    label_map={"true": 0, "false": 1},
    source_name="twitter",
    dataset_name="test_tsv"
)

dataset_counts = {
    "fake_news_test": len(fake_news_test),
    "fake_news_train": len(fake_news_train),
    "train": len(train),
    "train1": len(train1),
    "true_news": len(true_news),
    "fake_news_all": len(fake_news_all),
    "train_tsv": len(train_tsv),
    "test_tsv": len(test_tsv),
}
print(pd.Series(dataset_counts).sort_index())

big_dataset = pd.concat(
    [
        fake_news_test,
        fake_news_train,
        train,
        train1,
        true_news,
        fake_news_all,
        train_tsv,
        test_tsv,
    ],
    ignore_index=True
)

big_dataset = big_dataset.drop_duplicates(subset=["text"]).reset_index(drop=True)

less_big_dataset = sample_balanced_by_dataset(big_dataset, target_per_label=7500, random_state=42)

print(big_dataset["source_dataset"].value_counts())
print(big_dataset["dataset_name"].value_counts())
print(big_dataset["label"].value_counts())
print(less_big_dataset["source_dataset"].value_counts())
print(less_big_dataset["dataset_name"].value_counts())
print(less_big_dataset["label"].value_counts())
less_big_dataset.head(15000)


fake_news_all       17449
fake_news_test       1000
fake_news_train      5000
test_tsv              457
train                7503
train1             232003
train_tsv            3661
true_news           21190
dtype: int64
source_dataset
news       276642
twitter     11618
Name: count, dtype: int64
dataset_name
train1             232003
True                21190
Fake                17449
train                7503
fake_news_train      5000
train_tsv            3661
fake_news_test       1000
test_tsv              454
Name: count, dtype: int64
label
0    194016
1     94244
Name: count, dtype: int64
source_dataset
news       14284
twitter      716
Name: count, dtype: int64
dataset_name
train1             11721
Fake                1389
True                 819
train                465
fake_news_train      296
train_tsv            223
fake_news_test        59
test_tsv              28
Name: count, dtype: int64
label
1    7500
0    7500
Name: count, dtype: int64


,text,label,source_dataset,dataset_name
0,"PPG Industries Inc * PPG Electronics, Inc. * S...",1,news,train1
1,Florian Mayer put more than a year of injury p...,0,news,train1
2,States are kicking a growing number of people ...,1,news,train1
3,Realize power system system teacher here first...,0,news,fake_news_train
4,Snyder's-lance Inc * Snyder's-Lance begins per...,0,news,train1
...,...,...,...,...
14995,"Walgreens (yes, that Walgreens -- the one that...",0,news,train1
14996,Jabra's still probably best known for its head...,1,news,train1
14997,") DUBAI, July 24 (Reuters) - The Abu Dhabi Nat...",0,news,train1
14998,Represent carry stop former happy relate. Rela...,0,news,fake_news_train


In [7]:
less_big_dataset.to_csv('Data/smaller_data.csv', index=False)
big_dataset.to_csv('Data/big_data.csv', index=False)